<a href="https://colab.research.google.com/github/intisariapps-com/intiVoice_Studio/blob/main/intiVoice_Studio_Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎙️ intiVoice AI — Backend GPU Server (API Edition V1.8.0)

> ⏰ **Terakhir Diperbarui:** 22 September 2026 | **Rilis:** V1.8.0 (Headless GPU Server Edition)
> 🛡️ **Anti-Timeout Cloudflare:** Asynchronous Job Queue (`/job/create` & `/job/status`)
> 🚀 **Zero Pop-Up Google Drive:** Server berjalan instan tanpa dialog izin akses Drive (100% NVMe SSD Cepat).
> 🔐 **Protected Core Engine:** Modul mandiri terenkripsi resmi (Zero-Pip & 100% Kebal Cache).

Notebook ini berfungsi khusus sebagai **Engine Backend GPU Serverless** bertenaga Tesla T4 untuk melayani koneksi API dari **Web Studio Cloudflare Pages** atau **Aplikasi Desktop**.

### ✨ Fitur Utama V1.8.0:
1. 👥 **Dual-Speaker Dialogue Engine**: Sintesis percakapan dua pembicara dengan kloning suara masing-masing.
2. 🎭 **Natural Fluent Dialogue**: 13 kamus emosi vokal fungsional stabil.
3. ⏱️ **Live Countdown Timer Watchdog**: Pengaman hemat GPU dengan auto-shutdown cerdas.
4. 🔒 **Voice Anchor DNA System**: Mengunci kestabilan timbre suara di naskah panjang.
5. 🗣️ **Human-Cadence Sentence Chunking**: Pemenggalan naskah alami (180–240 karakter).
6. 🌬️ **Adaptive Breath Pause**: Jeda bernafas manusiawi antar kalimat (titik: 0.50s, koma: 0.25s).
7. ⚡ **Native Speed Control**: Kontrol tempo bicara alami via prompt instruction.
8. 🛡️ **Torchaudio Smart Sinc Resampler**: Preprocessing audio referensi 16kHz mono bebas distorsi.
9. 🎙️ **Ultimate & Controllable Voice Cloning**: Kloning vokal 1:1 akurat via referensi WAV + transcript.
10. 🎬 **Dual Subtitle CapCut Generator**: Otomatis membuat .SRT & .ASS Karaoke.
11. 🌐 **Cloudflare Permanent Named Tunnel**: Mendukung URL tetap permanen via token tunnel.

---
### 🚀 Cara Menjalankan:
1. Klik tombol **Play (▶)** di sel kode di bawah ini.
2. (Opsional) Masukkan **Cloudflare Tunnel Token** jika ingin domain permanen.
3. Tunggu hingga proses memuat model selesai dan URL Cloudflare Tunnel muncul di layar.
4. Salin URL tersebut ke aplikasi Web Studio pada **Tab Pengaturan Mesin** atau Aplikasi Desktop.


In [ ]:
"""
🎙️ INTIVOICE AI — BACKEND GPU SERVER (V1.8.0 PROTECTED RELEASE)
Hak Cipta (C) 2026 IntisariApps.com. Seluruh hak cipta dilindungi.
Fungsi: Engine API Backend Headless untuk Cloudflare Pages & Desktop App.
"""

# @title ⚙️ PUSAT KENDALI BACKEND GPU SERVER INTIVOICE
# @markdown Atur parameter sesi Colab di bawah ini:
AUTO_SHUTDOWN_MINUTES = 5  # @param [0, 1, 2, 3, 5, 10, 15, 30] {type:"raw"}
GOOGLE_DRIVE_MODE = "oauth_broker"  # @param ["oauth_broker", "native_mount", "disabled"]
DRIVE_BROKER_TOKEN = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🌐 Terowongan Publik & Cloudflare Named Tunnel (Opsional)
CLOUDFLARE_TUNNEL_TOKEN = ""  # @param {type:"string"}
CLOUDFLARE_TUNNEL_DOMAIN = ""  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}

import os
import sys
# [Colab Kernel Guardian] Proteksi Mutlak Anti-Heap-Corruption (T4 GPU Stabilizer)
os.environ["MALLOC_TRIM_THRESHOLD_"] = "100000"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["GOOGLE_DRIVE_MODE"] = str(GOOGLE_DRIVE_MODE).lower().strip() if "GOOGLE_DRIVE_MODE" in globals() else "oauth_broker"

import time
import subprocess
import zipfile
import sysconfig

print("=" * 80)
print("🚀 MEMULAI INTIVOICE AI BACKEND GPU SERVER V1.8.0 (PROTECTED)")
print("=" * 80)

# Penanganan Mode Google Drive
if GOOGLE_DRIVE_MODE == "native_mount":
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            print("⏳ Menghubungkan ke Google Drive pribadi Anda...")
            drive.mount("/content/drive")
        os.makedirs("/content/drive/MyDrive/intiVoice_Studio/clones", exist_ok=True)
        print("✅ [Google Drive] Native mount aktif di MyDrive/intiVoice_Studio/clones")
    except Exception as e_drv:
        print(f"⚠️ [Google Drive] Native mount dilewati: {e_drv}")
elif GOOGLE_DRIVE_MODE == "oauth_broker":
    broker_tok = ""
    if "DRIVE_BROKER_TOKEN" in globals() and DRIVE_BROKER_TOKEN and DRIVE_BROKER_TOKEN.strip():
        broker_tok = DRIVE_BROKER_TOKEN.strip()
    else:
        try:
            from google.colab import userdata
            broker_tok = (userdata.get("DRIVE_BROKER_TOKEN") or userdata.get("INTIVOICE_DRIVE_BROKER_TOKEN") or userdata.get("AUTOCUT_DRIVE_BROKER_TOKEN") or "").strip()
        except Exception:
            pass
    if broker_tok:
        os.environ["INTIVOICE_DRIVE_BROKER_TOKEN"] = broker_tok
        os.environ["DRIVE_BROKER_TOKEN"] = broker_tok
        print("✅ [Google Drive] Token tersimpan terdeteksi! Google Drive Vault terhubung otomatis.")
    else:
        print("✅ [Google Drive] Mode Brankas Cloud OAuth PKCE Aktif (Zero-FUSE, Bebas Freeze).")
else:
    print("ℹ️ [Google Drive] Integrasi Cloud Dinonaktifkan (Penyimpanan Lokal Saja).")

# [1/4] Download binary cloudflared jika belum ada
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⏳ Mengunduh Cloudflare Tunnel client...")
    subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

# [2/4] Install dependensi sistem dasar
print("📦 [1/4] Memeriksa dependensi audio & server...")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "fastapi", "uvicorn", "soundfile", "torchaudio", "pydantic", "requests", "qrcode"], check=False)

# [3/4] Unduh & pasang modul engine terenkripsi resmi (Zero-Pip & 100% Kebal Cache)
print("🔐 [2/4] Mengunduh modul backend engine: intivoice_studio_engine.zip (V1.8.0)...")
pkg_url = f"https://raw.githubusercontent.com/intisariapps-com/intiVoice_Studio/main/intivoice_studio_engine.zip?t={int(time.time())}"
pkg_local = "/tmp/intivoice_studio_engine.zip"
subprocess.run(["wget", "-q", "--no-cache", "--no-cookies", "-O", pkg_local, pkg_url], check=True)
pkg_size_kb = round(os.path.getsize(pkg_local) / 1024, 1) if os.path.exists(pkg_local) else 0.0
print(f"📦 [3/4] Paket Biner Terpasang: intivoice_studio_engine.zip (Versi V1.8.0 | {pkg_size_kb} KB)")
site_pkg = sysconfig.get_paths()["purelib"]
with zipfile.ZipFile(pkg_local, "r") as zf:
    zf.extractall(site_pkg)

# Kebal Cache: Bersihkan registri modul Python agar selalu memuat modul baru dari disk
for mod in list(sys.modules.keys()):
    if "intivoice_studio_engine" in mod or "pyarmor" in mod:
        del sys.modules[mod]

import intivoice_studio_engine
from intivoice_studio_engine import run_studio_server
engine_ver = getattr(intivoice_studio_engine, "__version__", "1.8.0")
print(f"🚀 [4/4] Modul Terverifikasi: intiVoice Studio Engine v{engine_ver} (Headless GPU Server)")

run_studio_server(
    tunnel_token=CLOUDFLARE_TUNNEL_TOKEN,
    tunnel_domain=CLOUDFLARE_TUNNEL_DOMAIN,
    hf_token=HF_TOKEN,
    auto_shutdown_minutes=AUTO_SHUTDOWN_MINUTES,
    mode="server",
    wait_forever=False
)
